In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path =  "/content/drive/MyDrive/Capstone_Project/student_learning performance.csvv"

In [ ]:
import pandas as pd
file_path =  "/content/drive/MyDrive/Capstone_Project/student_learning performance.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
df.info()
df.head()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
print("Duplicate Records:", df.duplicated().sum())
df.describe()

In [ ]:
for col in df.columns:
    print(f"\n{col}")
    print(df[col].nunique())

In [ ]:
df.columns = df.columns.str.lower()

df.columns

In [ ]:
cleaned_path = "/content/drive/MyDrive/Capstone_Project/student_learning performance.csv"

df.to_csv(cleaned_path, index=False)

print("Cleaned dataset saved successfully.")

In [ ]:
print("Rows Before:", len(df))
df_clean = df.drop_duplicates()
print("Rows After:", len(df_clean))

In [ ]:
df_clean.columns = df_clean.columns.str.lower().str.strip()
print(df_clean.columns.tolist())
print("Duplicates After Cleaning:", df_clean.duplicated().sum())

In [ ]:
cleaned_file = "/content/drive/MyDrive/student_cleaned.csv"

df_clean.to_csv(cleaned_file, index=False)

print("Cleaned dataset saved successfully.")

In [ ]:
print("Rows Before:", len(df))
print("Rows After:", len(df_clean))
print("Duplicates After Cleaning:", df_clean.duplicated().sum())

In [ ]:
import pandas as pd
import sqlite3
conn = sqlite3.connect('student_performance.db')

print("Database created successfully!")
df_clean.to_sql(
    'students',
    conn,
    if_exists='replace',
    index=False
)

print("Data loaded successfully!")

In [ ]:
query = """
SELECT COUNT(*) AS total_records
FROM students
"""

pd.read_sql(query, conn)

In [ ]:
query = """
SELECT *
FROM students
LIMIT 5
"""

pd.read_sql(query, conn)

In [ ]:
query = """
SELECT *
FROM students
WHERE examscore > 90
"""

result = pd.read_sql(query, conn)
result.head()

In [ ]:
query2 = """
SELECT *
FROM students
ORDER BY examscore DESC
LIMIT 10
"""

sorting_df = pd.read_sql(query2, conn)
sorting_df

In [ ]:
query3 = """
SELECT
    AVG(examscore) AS avg_exam_score,
    MAX(examscore) AS max_exam_score,
    MIN(examscore) AS min_exam_score
FROM students
"""

aggregation_df = pd.read_sql(query3, conn)
aggregation_df

In [ ]:
query4 = """
SELECT
    finalgrade,
    COUNT(*) AS student_count
FROM students
GROUP BY finalgrade
ORDER BY student_count DESC
"""

groupby_df = pd.read_sql(query4, conn)
groupby_df

In [ ]:
query5 = """
SELECT *
FROM students
WHERE attendance > 80
AND examscore > 85
"""

conditional_df = pd.read_sql(query5, conn)
conditional_df.head()

In [ ]:
query_transform = """
SELECT *,
CASE
    WHEN examscore >= 85 THEN 'Excellent'
    WHEN examscore >= 70 THEN 'Good'
    ELSE 'Needs Improvement'
END AS performance_category,

CASE
    WHEN attendance >= 90 THEN 'High Attendance'
    WHEN attendance >= 75 THEN 'Medium Attendance'
    ELSE 'Low Attendance'
END AS attendance_category

FROM students
"""

transformed_df = pd.read_sql(query_transform, conn)

transformed_df.head()

In [ ]:
transformed_df.to_csv(
    '/content/drive/MyDrive/Capstone_Project/student_learning performance.csv',
    index=False
)

print("Transformed dataset exported successfully!")

In [ ]:
import os

os.path.exists(
    '/content/drive/MyDrive/Capstone_Project/student_learning performance.csv'
)

In [ ]:
import pandas as pd

# Extract
df = pd.read_csv('/content/drive/MyDrive/Capstone_Project/student_learning performance.csv')

print("Data Extracted Successfully")
print(df.shape)

In [ ]:
# Remove duplicates
df_clean = df.drop_duplicates()

# Standardize column names
df_clean.columns = df_clean.columns.str.lower().str.strip()

print("Transformation Completed")
print(df_clean.shape)

In [ ]:
output_path = '/content/drive/MyDrive/Capstone_Project/student_learning performance.csv'

df_clean.to_csv(output_path, index=False)

print("Data Loaded Successfully")

In [ ]:
!pip install pyspark -q

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("StudentPerformanceMedallion") \
    .getOrCreate()

print("Spark Session Created Successfully")

In [ ]:
bronze_df = spark.read.csv(
    "/content/drive/MyDrive/student_transformed.csv",
    header=True,
    inferSchema=True
)

bronze_df.show(5)

In [ ]:
bronze_path = "/content/drive/MyDrive/bronze"

bronze_df.write \
    .mode("overwrite") \
    .parquet(bronze_path)

print("Bronze Layer Created Successfully")

In [ ]:
bronze_df.count()

In [ ]:
silver_df = spark.read.parquet(bronze_path)
silver_df = silver_df.dropDuplicates()
silver_df = silver_df.na.drop()
for col in silver_df.columns:
    silver_df = silver_df.withColumnRenamed(col, col.lower())
print("In silver layer the data cleaned")

In [ ]:
silver_df.printSchema()
silver_df.count()

In [ ]:
silver_path = "/content/drive/MyDrive/Capstone_Project/silver"

silver_df.write \
    .mode("overwrite") \
    .parquet(silver_path)

print("Silver Layer Created Successfully")

In [ ]:
from pyspark.sql import functions as F

gold_df = spark.read.parquet(silver_path)

In [ ]:
grade_performance = gold_df.groupBy("finalgrade") \
    .agg(
        F.avg("examscore").alias("avg_exam_score"),
        F.count("*").alias("student_count")
    )

grade_performance.show()

In [ ]:
age_performance = gold_df.groupBy("age") \
    .agg(
        F.avg("examscore").alias("avg_exam_score")
    )

age_performance.show()

In [ ]:
attendance_summary = gold_df.agg(
    F.avg("attendance").alias("avg_attendance"),
    F.max("attendance").alias("max_attendance"),
    F.min("attendance").alias("min_attendance")
)

attendance_summary.show()

In [ ]:
gold_path = "/content/drive/MyDrive/Capstone_Project/gold"

grade_performance.write \
    .mode("overwrite") \
    .parquet(f"{gold_path}/grade_performance")

age_performance.write \
    .mode("overwrite") \
    .parquet(f"{gold_path}/age_performance")

attendance_summary.write \
    .mode("overwrite") \
    .parquet(f"{gold_path}/attendance_summary")

print("Gold Layer Created Successfully")

In [ ]:
import os

csv_file = "/content/drive/MyDrive/Capstone_Project/student_learning performance.csv"

csv_size = os.path.getsize(csv_file) / (1024 * 1024)

print(f"CSV File Size: {csv_size:.2f} MB")

In [ ]:
parquet_folder = "/content/drive/MyDrive/Capstone_Project/bronze"

total_size = 0

for root, dirs, files in os.walk(parquet_folder):
    for file in files:
        total_size += os.path.getsize(os.path.join(root, file))

parquet_size = total_size / (1024 * 1024)

print(f"Parquet Size: {parquet_size:.2f} MB")

In [ ]:
gold_df = spark.read.parquet(
    "/content/drive/MyDrive/Capstone_Project/silver"
)

gold_df.show(5)

In [ ]:
pandas_df = gold_df.toPandas()

print(pandas_df.shape)

pandas_df.head()

In [ ]:
print("PySpark Rows:", gold_df.count())
print("Pandas Rows:", len(pandas_df))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

grade_pd = grade_performance.toPandas()

plt.figure(figsize=(8,5))

plt.bar(
    grade_pd['finalgrade'],
    grade_pd['avg_exam_score']
)

plt.title("Average Exam Score by Final Grade")
plt.xlabel("Final Grade")
plt.ylabel("Average Exam Score")

plt.show()

In [ ]:
grade_count = pandas_df['finalgrade'].value_counts()

plt.figure(figsize=(8,8))

plt.pie(
    grade_count,
    labels=grade_count.index,
    autopct='%1.1f%%'
)

plt.title("Final Grade Distribution")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.hist(
    pandas_df['attendance'],
    bins=20
)

plt.title("Attendance Distribution")
plt.xlabel("Attendance")
plt.ylabel("Frequency")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    pandas_df['attendance'],
    pandas_df['examscore']
)

plt.title("Attendance vs Exam Score")
plt.xlabel("Attendance")
plt.ylabel("Exam Score")

plt.show()

In [ ]:
import os

base_path = "/content/drive/MyDrive/Capstone_Project"

for root, dirs, files in os.walk(base_path):
    print(root)

In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/Capstone_Project/student_cleaned.csv')

In [ ]:
files.download('student_performance.db')

In [ ]:
import os

parquet_folder = "/content/drive/MyDrive/Capstone_Project/silver"

for file in os.listdir(parquet_folder):
    if file.endswith(".parquet"):
        print(file)

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/Capstone_Project/silver/part-00000-951e1e54-5d64-409f-ae2e-504b4447a091-c000.snappy.parquet"
)